# RigSafe — Level 2: Train a PPE detector

Goal: teach YOLO to see **helmet / vest / gloves / boots / goggles** and their **missing** versions
(`no_helmet`, `no_gloves`, ...), so later levels can raise PPE-violation alerts.

**Before running:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.

| Step | What | Time on T4 |
|---|---|---|
| 0 | Setup + GPU check + Google Drive | 1 min |
| 1 | Download & explore dataset | 2 min |
| 2 | Train YOLO26s | ~30–40 min |
| 3 | Evaluate on the **test** split | 1 min |
| 4 | Try on your own video | 1–2 min |
| 5 | Download `best.pt` to your laptop | — |

Everything is saved to **Google Drive** (`MyDrive/rigsafe`), so a Colab disconnect does not lose your training.

> This notebook also runs on a laptop without GPU in a tiny *smoke-test* mode (1 epoch, 5% data) — useful to check the code, useless as a model.

## 0. Setup

In [ ]:
import sys, subprocess
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)

import torch
HAS_GPU = torch.cuda.is_available()
print("Colab:", IN_COLAB, "| GPU:", torch.cuda.get_device_name(0) if HAS_GPU else "none")

if IN_COLAB and not HAS_GPU:
    raise RuntimeError("No GPU! Runtime -> Change runtime type -> T4 GPU, then run again.")

In [ ]:
# ---- Training settings (the knobs you will experiment with) ----
if IN_COLAB:
    MODEL    = "yolo26s.pt"   # pretrained on COCO -> we fine-tune it (transfer learning)
    EPOCHS   = 60             # full passes over the training images
    IMGSZ    = 640            # input size; must match what you use at inference later
    BATCH    = 16             # images per step; fits a T4's 15 GB
    FRACTION = 1.0            # use 100% of the training set
    PATIENCE = 15             # stop early if val mAP doesn't improve for 15 epochs
else:  # laptop smoke test: only proves the code runs
    MODEL, EPOCHS, IMGSZ, BATCH, FRACTION, PATIENCE = "yolo26n.pt", 1, 320, 8, 0.05, 5

RUN_NAME = f"ppe_{Path(MODEL).stem}_{IMGSZ}"
DATA = "construction-ppe.yaml"   # built into Ultralytics; auto-downloads (~150-180 MB)

USE_DRIVE = True   # set False to skip Google Drive (results then vanish when the session ends!)

if IN_COLAB:
    PROJECT = Path("/content/rigsafe")   # fallback: Colab's temporary disk
    if USE_DRIVE:
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            PROJECT = Path("/content/drive/MyDrive/rigsafe")
        except Exception as err:  # e.g. "credential propagation was unsuccessful"
            print(f"WARNING: Drive mount failed ({err}).")
            print("Continuing on Colab's temporary disk. Download best.pt at the end (step 5) "
                  "before closing the tab. To retry Drive: tick ALL permission boxes, use one "
                  "Google account (try Incognito), then Runtime -> Restart session.")
else:
    PROJECT = Path("../outputs/l2_local").resolve()
PROJECT.mkdir(parents=True, exist_ok=True)
print("Saving runs to:", PROJECT)

## 1. Explore the dataset

**Never train blind.** Look at how many examples each class has and what the labels look like.
YOLO label format: one `.txt` per image, one line per box: `class x_center y_center width height` (all 0–1, relative to image size).

In [ ]:
from collections import Counter
from ultralytics.data.utils import check_det_dataset

data = check_det_dataset(DATA)          # downloads on first run
NAMES = data["names"]
print("Dataset root:", data["path"])
print("Classes:", NAMES)

def label_dir(split_images: str) -> Path:
    return Path(str(split_images).replace("/images/", "/labels/").replace("\\images\\", "\\labels\\"))

def count_split(split: str):
    labels = label_dir(data[split])
    files = list(labels.glob("*.txt"))
    boxes = Counter()
    for f in files:
        for line in f.read_text().splitlines():
            if line.strip():
                boxes[int(line.split()[0])] += 1
    return len(files), boxes

stats = {s: count_split(s) for s in ("train", "val", "test") if data.get(s)}
print(f"\n{'class':<12}" + "".join(f"{s:>8}" for s in stats))
for cid, name in NAMES.items():
    print(f"{name:<12}" + "".join(f"{stats[s][1][cid]:>8}" for s in stats))
print(f"{'IMAGES':<12}" + "".join(f"{stats[s][0]:>8}" for s in stats))

In [ ]:
import matplotlib.pyplot as plt

train_counts = [stats["train"][1][c] for c in NAMES]
plt.figure(figsize=(10, 3.5))
bars = plt.bar(list(NAMES.values()), train_counts,
               color=["tab:red" if n.startswith("no_") or n == "none" else "tab:green" for n in NAMES.values()])
plt.bar_label(bars)
plt.title("Training boxes per class (red = missing-PPE classes)")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
import random
import cv2

def show_samples(split="train", n=6, seed=0):
    images = sorted(Path(data[split]).glob("*.*"))
    random.Random(seed).shuffle(images)
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    for ax, img_path in zip(axes.flat, images[:n]):
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        lbl = label_dir(data[split]) / (img_path.stem + ".txt")
        for line in (lbl.read_text().splitlines() if lbl.exists() else []):
            c, xc, yc, bw, bh = line.split()[:5]
            c = int(c); xc, yc, bw, bh = map(float, (xc, yc, bw, bh))
            x1, y1 = int((xc - bw / 2) * w), int((yc - bh / 2) * h)
            x2, y2 = int((xc + bw / 2) * w), int((yc + bh / 2) * h)
            name = NAMES[c]
            color = (255, 0, 0) if name.startswith("no_") or name == "none" else (0, 200, 0)
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
            cv2.putText(img, name, (x1, max(12, y1 - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
        ax.imshow(img); ax.set_title(img_path.name, fontsize=8); ax.axis("off")
    plt.tight_layout(); plt.show()

show_samples()

**Questions to note down:**
1. Which classes have the fewest boxes? Expect the model to be worst at those.
2. Do the images look like *your* CCTV (high, far, wide) or like close-up photos? This gap is called **domain shift**.

## 2. Train

This is **transfer learning**: YOLO26s already knows edges, shapes and people from COCO (330k images).
We only teach it the new PPE classes, so ~1,100 images are enough.

Watch the printed table: `box_loss`/`cls_loss` should go **down**, `mAP50` should go **up**.

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)
model.train(
    data=DATA,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    fraction=FRACTION,
    patience=PATIENCE,
    project=str(PROJECT / "runs"),
    name=RUN_NAME,
    exist_ok=True,
    seed=0,
    plots=True,
    device=0 if HAS_GPU else "cpu",
    workers=2 if IN_COLAB else 0,
)
RUN_DIR = PROJECT / "runs" / RUN_NAME
BEST = RUN_DIR / "weights" / "best.pt"
print("Best weights:", BEST, "exists:", BEST.exists())

**Colab disconnected mid-training?** Re-run the two Setup cells, then this cell — it continues from the last saved epoch.

In [ ]:
RESUME = False   # set True only after a disconnect
if RESUME:
    YOLO(str(PROJECT / "runs" / RUN_NAME / "weights" / "last.pt")).train(resume=True)

## 3. Evaluate on the **test** split

Training used `train`, early-stopping watched `val`. The `test` images were never seen — this is the honest score.

- **Precision**: of the boxes the model drew, how many were right? (low → false alarms)
- **Recall**: of the real objects, how many did it find? (low → missed violations — **worse for safety**)
- **mAP50**: overall quality, box counts as correct if it overlaps the truth ≥50%
- **mAP50-95**: stricter, averages overlap thresholds 50%…95%

In [ ]:
best = YOLO(str(BEST))
metrics = best.val(data=DATA, split="test", imgsz=IMGSZ, batch=BATCH,
                   project=str(PROJECT / "runs"), name=RUN_NAME + "_test", exist_ok=True, plots=True)

box = metrics.box
print(f"\nTEST  mAP50={box.map50:.3f}  mAP50-95={box.map:.3f}  P={box.mp:.3f}  R={box.mr:.3f}\n")
print(f"{'class':<12}{'P':>7}{'R':>7}{'mAP50':>8}")
for i, cid in enumerate(box.ap_class_index):
    print(f"{NAMES[int(cid)]:<12}{box.p[i]:>7.2f}{box.r[i]:>7.2f}{box.ap50[i]:>8.2f}")

In [ ]:
from IPython.display import Image, display

for name in ("results.png", "confusion_matrix_normalized.png"):
    p = RUN_DIR / name
    if p.exists():
        print(name); display(Image(filename=str(p), width=900))
    else:
        print("missing (normal in smoke-test mode):", name)

**How to read them:**
- `results.png`: train & val losses falling together = good. Val loss rising while train falls = **overfitting**.
- Confusion matrix: the diagonal is correct predictions. The `background` row/column = missed objects / false boxes.
  Look for `no_helmet` being confused with `helmet` — that is the error that matters most for us.

## 4. Try it on your own CCTV-style video

In [ ]:
if IN_COLAB:
    from google.colab import files
    print("Choose a short video (< 100 MB) from your laptop, e.g. data/videos/site.mp4")
    uploaded = files.upload()
    VIDEO = next(iter(uploaded))
else:
    VIDEO = "../data/videos/site.mp4"

for _ in best.predict(source=VIDEO, conf=0.3, imgsz=IMGSZ, stream=True, verbose=False,
                      save=True, project=str(PROJECT / "runs"), name=RUN_NAME + "_video", exist_ok=True):
    pass   # stream=True processes frame by frame without filling RAM
print("Annotated video saved in:", PROJECT / "runs" / (RUN_NAME + "_video"))

## 5. Bring the model to your laptop

In [ ]:
import shutil

final = PROJECT / f"{RUN_NAME}_best.pt"
shutil.copy(BEST, final)
print("Copied to:", final)
if IN_COLAB:
    from google.colab import files
    files.download(str(final))

Then on your laptop:

```bash
mv ~/Downloads/ppe_yolo26s_640_best.pt models/ppe_best.pt
./scripts/l2_detect_ppe.py --source data/videos/site.mp4 --model models/ppe_best.pt --show
```

### Record in your notes
| Run | Model | Epochs | imgsz | test mAP50 | Recall | no_helmet AP50 |
|---|---|---|---|---|---|---|
| 1 | yolo26s | 60 | 640 | | | |